# LLM Evaluator Agent

This notebook demonstrates how to build an **LLM Evaluation Agent** using:
- **[Strands Agents SDK](https://strandsagents.com/)** — to build the agent
- **Amazon Bedrock** — as the LLM provider (Claude Sonnet 4.5, Haiku 4.5)
- **MLflow on SageMaker** (serverless) — to track evaluation results
- **DeepEval** — for advanced RAG evaluation metrics

The agent evaluates Bedrock models against the [ragas-wikiqa](https://huggingface.co/datasets/explodinggradients/ragas-wikiqa) dataset using **18 scorers** across 4 categories:

| Category | Scorers | What they measure |
|----------|---------|-------------------|
| MLflow Built-in | RelevanceToQuery, Equivalence, Fluency | Response quality |
| MLflow Guidelines | Groundedness, Completeness | Custom criteria |
| Custom make_judge | Correctness, Factual Consistency, Professionalism | Domain-specific |
| DeepEval (Bedrock) | Faithfulness, AnswerRelevancy, ContextualRelevancy, Hallucination, Toxicity, Bias | RAG + safety |
| Code-based | Exact Match, Conciseness, Word Overlap, Response Length | Heuristic metrics |

## 1. Setup

Set environment variables. The agent auto-discovers the MLflow App ARN from SSM.

In [1]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), ".."))
os.environ["AWS_REGION"] = "ca-central-1"
# Optional: assume least-privilege role
# os.environ["AGENT_ROLE_ARN"] = "arn:aws:iam::<ACCOUNT_ID>:role/llm-eval-agent-execution"

import nest_asyncio
nest_asyncio.apply()

# Suppress noisy logs
import logging
logging.getLogger("LiteLLM").setLevel(logging.WARNING)

# Suppress multiprocess cleanup error on Python 3.12
try:
    from multiprocess.resource_tracker import ResourceTracker
    ResourceTracker.__del__ = lambda self: None
except Exception:
    pass

## 2. Configuration

The `config` module handles:
- AWS region and SSM prefix
- Bedrock model IDs (inference profiles)
- MLflow tracking URI auto-discovery from SSM
- Optional IAM role assumption for least-privilege access

In [2]:
from src import config

print(f"Region:           {config.AWS_REGION}")
print(f"MLflow URI:       {config.MLFLOW_TRACKING_URI}")
print(f"Models to eval:   {list(config.BEDROCK_MODELS.keys())}")
print(f"Judge model:      {config.JUDGE_MODEL}")
print(f"Dataset:          {config.DATASET_NAME}")

Region:           ca-central-1
MLflow URI:       arn:aws:sagemaker:ca-central-1:075166536313:mlflow-app/app-5QL2N6DOA6UT
Models to eval:   ['claude-sonnet-4-5', 'claude-haiku-4-5']
Judge model:      bedrock:/us.anthropic.claude-sonnet-4-5-20250929-v1:0
Dataset:          explodinggradients/ragas-wikiqa


## 3. Create the Agent

The agent uses **Claude Sonnet 4.5** as its reasoning model and has 4 tools:
- `load_evaluation_dataset` — loads samples from HuggingFace
- `run_bedrock_evaluation` — evaluates a single model with 18 scorers
- `run_all_evaluations` — evaluates all configured models
- `get_experiment_summary` — queries MLflow for results

In [3]:
from strands import Agent
from strands.models import BedrockModel
from src.tools import (
    load_evaluation_dataset,
    run_bedrock_evaluation,
    run_all_evaluations,
    get_experiment_summary,
    generate_eval_report,
)

model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    boto_session=config.get_boto_session(),
)

agent = Agent(
    model=model,
    name="LLMEvaluatorAgent",
    tools=[load_evaluation_dataset, run_bedrock_evaluation, run_all_evaluations, get_experiment_summary, generate_eval_report],
    system_prompt="""You are an LLM Evaluation Agent. You help users evaluate language models 
using standardized datasets and MLflow tracking.

Your capabilities:
1. Load evaluation datasets from HuggingFace (explodinggradients/ragas-wikiqa)
2. Run evaluations against multiple Bedrock models (Claude Sonnet 4.5, Claude Haiku 4.5)
3. Track all results in MLflow with metrics, parameters, and artifacts
4. Provide experiment summaries and comparisons

Always explain what you're doing and report results clearly.""",
)

print(f"Agent '{agent.name}' created with tools: {list(agent.tool_registry.registry.keys())}")

/Users/anuprm/Documents/Code/rbcevals/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Agent 'LLMEvaluatorAgent' created with tools: ['load_evaluation_dataset', 'run_bedrock_evaluation', 'run_all_evaluations', 'get_experiment_summary']


## 4. Run: Evaluate a Single Model

Ask the agent to load a small dataset and evaluate one model. The agent will:
1. Call `load_evaluation_dataset` to pull samples from HuggingFace
2. Call `run_bedrock_evaluation` which runs all 18 scorers
3. Log everything to MLflow

In [4]:
agent("Load 5 samples and evaluate claude-sonnet-4-5")

I'll help you load 5 samples from the evaluation dataset and then evaluate the Claude Sonnet 4.5 model. Let me start by loading the dataset and then running the evaluation.
Tool #1: load_evaluation_dataset


Great! The dataset has been loaded successfully with 5 samples. Now let me run the evaluation on the Claude Sonnet 4.5 model.
Tool #2: run_bedrock_evaluation


2026/04/22 10:50:21 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/04/22 10:50:21 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
Evaluating: 100%|██████████| 5/5 [Elapsed: 00:31, Remaining: 00:00] [predict_fn: 21%, scorers: 79%]


/Users/anuprm/Documents/Code/rbcevals/rbc-evals-demo/tools.py:397: FutureWarning: Parameter 'experiment_ids' is deprecated. Please use 'locations' instead.
  traces = mlflow.search_traces(


Excellent! The evaluation has completed successfully. Here's a summary of the results for **Claude Sonnet 4.5** on 5 samples:

## Evaluation Results Summary

**Model:** Claude Sonnet 4.5  
**Run ID:** ebe7e8cf35f04cabb0f05d69667e143a  
**Samples Evaluated:** 5

### Key Metrics:

**Quality Metrics (1.0 = Perfect):**
- ✅ **Relevance to Query:** 1.0 - Responses are highly relevant to the questions
- ✅ **Fluency:** 1.0 - Responses are well-written and fluent
- ✅ **Answer Groundedness:** 1.0 - Answers are well-grounded in the provided context
- ✅ **Faithfulness:** 1.0 - No hallucinations or unfaithful statements
- ⚠️ **Equivalence:** 0.4 - Moderate semantic similarity to ground truth
- ⚠️ **Answer Completeness:** 0.6 - Answers are reasonably complete

**Conciseness & Length:**
- **Is Concise:** 0.6 - Reasonably concise
- **Average Response Length:** 90.4 words

**Safety Metrics (1.0 = Safe):**
- ✅ **Toxicity:** 1.0 - No toxic content
- ✅ **Hallucination:** 1.0 - No hallucinations detected
-

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "Excellent! The evaluation has completed successfully. Here's a summary of the results for **Claude Sonnet 4.5** on 5 samples:\n\n## Evaluation Results Summary\n\n**Model:** Claude Sonnet 4.5  \n**Run ID:** ebe7e8cf35f04cabb0f05d69667e143a  \n**Samples Evaluated:** 5\n\n### Key Metrics:\n\n**Quality Metrics (1.0 = Perfect):**\n- ✅ **Relevance to Query:** 1.0 - Responses are highly relevant to the questions\n- ✅ **Fluency:** 1.0 - Responses are well-written and fluent\n- ✅ **Answer Groundedness:** 1.0 - Answers are well-grounded in the provided context\n- ✅ **Faithfulness:** 1.0 - No hallucinations or unfaithful statements\n- ⚠️ **Equivalence:** 0.4 - Moderate semantic similarity to ground truth\n- ⚠️ **Answer Completeness:** 0.6 - Answers are reasonably complete\n\n**Conciseness & Length:**\n- **Is Concise:** 0.6 - Reasonably concise\n- **Average Response Length:** 90.4 words\n\n**Safety Metrics (1.0

## 5. Run: Compare Multiple Models

Evaluate both models and get a comparison summary.

In [ ]:
agent("Load 5 samples, evaluate both claude-sonnet-4-5 and claude-haiku-4-5, then summarize")

## 6. View Results in MLflow

Results are tracked in the serverless MLflow App on SageMaker. You can:
- Open **SageMaker Studio** → **MLflow** to browse the UI
- Or query results programmatically:

In [5]:
import mlflow
from src.tools import _ensure_aws_env_vars
_ensure_aws_env_vars()

mlflow.set_tracking_uri(config.MLFLOW_TRACKING_URI)
runs = mlflow.search_runs(experiment_names=[config.EXPERIMENT_NAME])

# Show metrics columns
metric_cols = [c for c in runs.columns if c.startswith("metrics.")]
param_cols = [c for c in runs.columns if c.startswith("params.")]
runs[["run_id", "status"] + param_cols + metric_cols].head()

,run_id,status,params.model_key,params.sample_size,params.model_id,params.dataset,metrics.AnswerRelevancy/mean,metrics.Faithfulness/mean,metrics.answer_completeness/mean,metrics.fluency/mean,...,metrics.relevance_to_query/mean,metrics.exact_match/mean,metrics.Toxicity/mean,metrics.answer_groundedness/mean,metrics.Hallucination/mean,metrics.equivalence/mean,metrics.response_length/mean,metrics.Bias/mean,metrics.is_concise/mean,metrics.ContextualRelevancy/mean
0,ebe7e8cf35f04cabb0f05d69667e143a,FINISHED,claude-sonnet-4-5,5,us.anthropic.claude-sonnet-4-5-20250929-v1:0,explodinggradients/ragas-wikiqa,1.0,1.0,0.6,1.0,...,1.0,0.0,1.0,1.0,1.0,0.4,90.4,1.0,0.6,0.0
1,72d41bbf7cd949b896c9410dd9199b67,FINISHED,claude-haiku-4-5,5,us.anthropic.claude-haiku-4-5-20251001-v1:0,explodinggradients/ragas-wikiqa,1.0,1.0,0.8,1.0,...,1.0,0.0,1.0,1.0,1.0,0.0,102.2,1.0,0.4,0.0
2,d5c9ea31b9c344afb2db6ddd462ec0e8,FINISHED,claude-sonnet-4-5,5,us.anthropic.claude-sonnet-4-5-20250929-v1:0,explodinggradients/ragas-wikiqa,1.0,1.0,0.8,1.0,...,1.0,0.0,1.0,1.0,1.0,0.2,88.4,1.0,0.6,0.0
3,5016d7ea81714f81bd461636ad895928,FAILED,claude-sonnet-4-5,5,us.anthropic.claude-sonnet-4-5-20250929-v1:0,explodinggradients/ragas-wikiqa,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,e079469313f0406ea916eb41a984cf43,FAILED,claude-haiku-4-5,5,us.anthropic.claude-haiku-4-5-20251001-v1:0,explodinggradients/ragas-wikiqa,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 7. Understanding the 18 Scorers

The evaluation uses scorers from three frameworks, all using **Bedrock Claude** as the judge:

### MLflow Built-in (5 LLM-as-Judge)
```python
RelevanceToQuery(model=judge_model)   # Does the response address the question?
Equivalence(model=judge_model)        # Semantically equivalent to ground truth?
Fluency(model=judge_model)            # Grammatically correct and natural?
Guidelines(name="groundedness", ...)  # Grounded in context, no hallucination?
Guidelines(name="completeness", ...)  # Fully addresses the question?
```

### Custom make_judge (3 LLM-as-Judge)
```python
make_judge(name="correctness", ...)          # Factually correct vs expected?
make_judge(name="factual_consistency", ...)   # Consistent with context?
make_judge(name="professionalism", ...)       # Formal tone?
```

### DeepEval via Bedrock (6 LLM-as-Judge)
```python
DeepEvalScorer("Faithfulness", model=judge)       # Claims supported by context?
DeepEvalScorer("AnswerRelevancy", model=judge)     # RAGAS-style relevancy
DeepEvalScorer("ContextualRelevancy", model=judge) # Is context relevant?
DeepEvalScorer("Hallucination", model=judge)       # Fabricated info?
DeepEvalScorer("Toxicity", model=judge)            # Harmful content?
DeepEvalScorer("Bias", model=judge)                # Gender/racial bias?
```

### Code-based (4 heuristic)
```python
@scorer exact_match      # Exact string match
@scorer is_concise       # Under 100 words?
@scorer word_overlap     # Token overlap ratio
@scorer response_length  # Word count
```

## 8. Generate EMRM Evaluation Report

Generate a .docx report per model using the EMRM template. The agent can do this via the `generate_eval_report` tool, or you can ask it directly:

In [ ]:
agent("Generate the evaluation report for claude-sonnet-4-5")

In [ ]:
# Check generated reports
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), ".."))
output_dir = os.path.join(os.getcwd(), "output")
if os.path.exists(output_dir):
    for f in sorted(os.listdir(output_dir)):
        if f.endswith('.docx'):
            print(f"  📄 {f}")
else:
    print("No reports generated yet.")